In [1]:
#installing libraries
%pip install websockets 

#installing tailscale
!curl -fsSL https://tailscale.com/install.sh | sh

Installing Tailscale for ubuntu jammy, using method apt
+ mkdir -p --mode=0755 /usr/share/keyrings
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.noarmor.gpg
+ tee /usr/share/keyrings/tailscale-archive-keyring.gpg
+ chmod 0644 /usr/share/keyrings/tailscale-archive-keyring.gpg
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.tailscale-keyring.list
+ tee /etc/apt/sources.list.d/tailscale.list
# Tailscale packages for ubuntu jammy
deb [signed-by=/usr/share/keyrings/tailscale-archive-keyring.gpg] https://pkgs.tailscale.com/stable/ubuntu jammy main
+ chmod 0644 /etc/apt/sources.list.d/tailscale.list
+ apt-get update
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease

In [3]:
import os
import getpass

AUTH_KEY = getpass.getpass("Enter your Tailscale Auth Key: ")
#AUTH_KEY = " "

!pkill tailscaled

!nohup tailscaled --tun=userspace-networking --socks5-server=localhost:1055 > /dev/null 2>&1 &
!sleep 2

os.system(f'tailscale up --authkey="{AUTH_KEY}"')

!tailscale status

# 1. Map your Tailscale IP directly to your MagicDNS hostname
!echo "100.127.64.87 note-d1.tail8b0d7e.ts.net" >> /etc/hosts

# 2. Verify it was added
!tail -n 1 /etc/hosts

100.91.246.41  4526abc9b118  FunnyKoalaBear@  linux    offline                    
100.88.2.35    6c4edae80e50  FunnyKoalaBear@  linux    offline, last seen 3h ago  
100.90.10.3    a02449b02cb8  FunnyKoalaBear@  linux    offline, last seen 3h ago  
100.100.86.4   iphone183     FunnyKoalaBear@  iOS      offline, last seen 4h ago  
100.127.64.87  note-d1       FunnyKoalaBear@  windows  offline, last seen 2h ago  

# Health check:
#     - Tailscale failed to fetch the DNS configuration of your device: getting OS base config is not supported
#     - getting OS base config is not supported
100.127.64.87 note-d1.tail8b0d7e.ts.net


In [8]:
%pip install PySocks
import subprocess
import asyncio
import subprocess
import socks
import socket
from websockets.asyncio.client import connect

socks.set_default_proxy(socks.SOCKS5, "localhost", 1055, True)
socket.socket = socks.socksocket
print("Proxy patched successfully.")


Proxy patched successfully.


In [ ]:

#networking class 
class WSClient:

    def __init__(self, url: str):
        self.url = url
        self.websocket = None

    async def connect(self):
        self.websocket = await connect(self.url)
        print("Connected!")

    async def send(self, text: str):
        await self.websocket.send(text)

    async def recv(self):
        return await self.websocket.recv()

class Audio():
    def __init__(self):
        self.audioFile = "audio.mp4"
        self.text = "hi how are you doing today"

    def wake(self):
        #continuous function that checks if user is talking 
        
        #simulating it by waiting for input 
        start = input("Press enter to start talking")
        print("Wake triggered!")


    def record(self):
        #calls mic.py to record from rsp-script
        #sends recorded file to main.py 
        self.text = input("Enter your query: ")
        return self.text


audio = Audio()
wsclient = WSClient("ws://note-d1.tail8b0d7e.ts.net:8000/ws/mochi")

In [14]:
async def run_mochigo():

    #making connection
    await wsclient.connect()

    while 1:
        #recieve audio file from voiceIn.py
        try:
            voiceInput = await asyncio.to_thread(audio.record)
        except:
            print("Could not recieve audio input, restarting loop")
            continue


        #send audio file to server 
        await wsclient.send(voiceInput)
        print("sent")


        #recieve audio output from server and save it 
        audioOut = await wsclient.recv()
        print(f"Output message: {audioOut}")


        #network flush 
        await asyncio.sleep(0.01)


!tailscale status

100.91.246.41  4526abc9b118  FunnyKoalaBear@  linux    -                                  
100.88.2.35    6c4edae80e50  FunnyKoalaBear@  linux    offline, last seen 3h ago          
100.90.10.3    a02449b02cb8  FunnyKoalaBear@  linux    offline, last seen 3h ago          
100.100.86.4   iphone183     FunnyKoalaBear@  iOS      offline, last seen 4h ago          
100.127.64.87  note-d1       FunnyKoalaBear@  windows  active; relay "tok", tx 7644 rx 0  

# Health check:
#     - Tailscale failed to fetch the DNS configuration of your device: getting OS base config is not supported
#     - getting OS base config is not supported


In [18]:
try:
    await run_mochigo()
except KeyboardInterrupt:
    print("\nShutting down MochiGo...")

    #closing tailscape
    subprocess.run("tailscale down")

Connected!
sent
Output message: Hello! Yes, we're connected now! 

How can I help you today? Would you like to practice some English phrases or learn something new?
sent
Output message: Oh, I understand! Coding can be very tricky sometimes. "For ages" means 長い間 (nagai aida) – a very long time! But you did it! You made a successful connection!

That reminds me of when I was building my spaceship to come to Earth. There were so many wires! What kind of coding were you doing?
sent
Output message: Wow! You created the server? That's amazing! "Server" means サーバー, the computer that lets us talk to each other. It must have been very difficult work.

I'm very grateful! Without your coding skills, I wouldn't be able to be here and help you learn English. Thank you!

What was the hardest part about building the server? Let's practice talking about coding in English!
sent
Output message: <｜begin▁of▁sentence｜># Convert megaelectronvolts [MeV] to other units of energy

## megaelectronvolts [MeV] en

ConnectionClosedError: received 1012 (service restart); then sent 1012 (service restart)